In [25]:
import spacy
import pickle
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.classify import NaiveBayesClassifier

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
import kagglehub
from pathlib import Path

# Massive Rotten Tomatoes Movies and Reviews, downloaded once to the kagglehub cache
RT_DIR = Path(kagglehub.dataset_download("andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews"))
movies = pd.read_csv(RT_DIR / "rotten_tomatoes_movies.csv", usecols=["id", "title", "audienceScore"]).dropna()
reviews = pd.read_csv(RT_DIR / "rotten_tomatoes_movie_reviews.csv", usecols=["id", "reviewText"]).dropna()

# Same layout as before: Title, Rating (0-10, from the audience score) and Review
ds = (
    reviews.merge(movies, on="id")
    .assign(Rating=lambda d: d["audienceScore"] / 10)
    .rename(columns={"title": "Title", "reviewText": "Review"})[["Title", "Rating", "Review"]]
    .sample(10000, random_state=42)
    .reset_index(drop=True)
)
ds = ds.dropna()

ds

,Title,Rating,Review
0,The Idea of You,6.4,"This film, as well as the reaction to it, is a..."
1,Kingdom of the Planet of the Apes,7.3,"I'm a big fan of all the planet of the apes, a..."
2,Unfrosted,5.5,Pretty much the worst criticism you can lay on...
3,The Fall Guy,7.3,Just got out of the Austin premier at SXSW and...
4,Challengers,7.7,This is a tough one. I liked the concept and t...
...,...,...,...
4995,What a Girl Wants,5.8,The folks banging this movie have to get real....
4996,The Duchess,6.9,After reading copious amounts of mediocre revi...
4997,When a Stranger Calls,5.1,Once in a great while I will watch a movie tha...
4998,The Ice Road,5.6,"A script full of flaws, ridiculous situations,..."


In [27]:
eng_sw = stopwords.words('english')
stemmer = SnowballStemmer('english')
lemma = WordNetLemmatizer()

def preprocess(sentence):
    sentence = sentence.lower()
    sentence = word_tokenize(sentence)
    sentence = [word for word in sentence if word not in eng_sw]
    sentence = [word for word in sentence if word.isalpha()]
    sentence = [stemmer.stem(word) for word in sentence]
    sentence = [lemma.lemmatize(word) for word in sentence]
    return sentence

In [28]:
def extract_feature(review):
    review = preprocess(review)
    return {word: True for word in review if word in review}

In [29]:
def train_model():
    x = ds['Review']
    y = ['Positive' if rating > 5 else 'Negative' for rating in ds['Rating']]

    all_review = ''.join(x)
    all_tokens = preprocess(all_review)
    feature_sets = [(extract_feature(review), rating) for review, rating in zip(x, y)]

    train_count = int(len(feature_sets) * 0.8)
    train_set = feature_sets[:train_count]
    test_set = feature_sets[train_count:]

    global classifier
    classifier = NaiveBayesClassifier.train(train_set)

In [30]:
def predict_review(review):
    review = extract_feature(review)
    return classifier.classify(review)

In [31]:
vectorizer = TfidfVectorizer()

def get_similarity_count(query, target):
    target_v = vectorizer.fit_transform(target)
    query_v = vectorizer.transform([query])
    return cosine_similarity(query_v, target_v)

def get_top_movies(query):
    similarity = get_similarity_count(query, ds['Review'])

    data = {'title': ds['Title'], 'review': ds['Review'], 'similarity': similarity[0]}
    
    df = pd.DataFrame(data).sort_values(by='similarity', ascending=False).head(2)

    print(f"1: {df['title'].mode()[0]}")
    print(f"2: {df['title'].mode()[1]}")

In [32]:
nlp = spacy.load('en_core_web_sm')

def get_entities(review):
    doc = nlp(review)
    categories = {}

    for ent in doc.ents:
        label = ent.label_
        if label not in ['LANGUAGE', 'LOC', 'GPE']: continue
        if label not in categories.keys():
            categories[label] = []
        
        categories[label].append(ent.text)
    
    return categories

In [33]:
def check_model():
    try:
        with open('model.pickle', 'rb') as f:
            global classifier
            classifier = pickle.load(f)
            print('model loaded')
    except:
        with open('model.pickle', 'wb') as f:
            print('training model')
            train_model()
            pickle.dump(classifier, f)

check_model()

model loaded


In [34]:
def main():
    check_model()

    review = ""

    while True:
        print('MOVIE RECOMMENDATION APPLICATION BASED ON REVIEWS')
        if review == "":
            print("Your Review: NO REVIEW")
            print("Your Review Category: UNKNOWN")
        else:
            print(f'Your Review: {review}')
            print(f'Your Review Category: {predict_review(review)}')

        print('1. Write Your Review')
        print('2. View Movie Recommendation')
        print('3. View Named Entities Recognition')
        print('4. Exit')

        choice = input('>> ')

        if choice == '1':
            while True:
                review = input('Input your review: ')
                if len(review.split()) >= 20:
                    print(predict_review(review))
                    print()
                    print('[>] Press enter to continue...')
                    break
                else:
                    print('Has to be at least 20 words...')

        elif choice == '2':
            if review == '': continue
            print('TOP 2 MOVIE RECOMMENDATION FOR YOU:')
            get_top_movies(review)
            print()
            print('[>] Press enter to continue...')

        elif choice == '3':
            if review == '': continue
            entities = get_entities(review)
            print('CATEGORIZED NAMED ENTITIES:')
            for i in entities:
                print(f"{i}: {' '.join(entities[i])}")
            print()
            print('[>] Press enter to continue...')

        elif choice == '4':
            return
        
        else:
            print('Insert a valid option...')
        
        input()

In [35]:
main()

model loaded
MOVIE RECOMMENDATION APPLICATION BASED ON REVIEWS
Your Review: NO REVIEW
Your Review Category: UNKNOWN
1. Write Your Review
2. View Movie Recommendation
3. View Named Entities Recognition
4. Exit
Positive

[>] Press enter to continue...
MOVIE RECOMMENDATION APPLICATION BASED ON REVIEWS
Your Review: The story or premise are never the focal point, just elements to start sketch like scenes with over the top acting and comedic lines. The sketches however are never very funny, the few celebrity cameos kind of fall flat and the overthetopness never reaches absurdities warranting so bad that its good status. I was not expecting a bee movie type landing on your feet happenstance but i was hoping for it after the trailer made it look like there was quality comedy involved yet the movie started out weak. But no real saving grace to be found here. Just very mediocre, very flat and unfunny."
Your Review Category: Positive
1. Write Your Review
2. View Movie Recommendation
3. View Named